# **Semantic Web & Linked Open Data with Python**
## บทที่ 11 — SWRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch11/ch11_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://www.w3.org/submissions/SWRL/
*   https://owlready2.readthedocs.io/

📖 เอกสารประกอบบทนี้: [`11-swrl/`](https://github.com/Rathachai/semantic-web/tree/main/11-swrl)

---

### Setup

In [1]:
!pip install -q owlready2

In [2]:
import shutil, subprocess

if shutil.which('java') is None:
    print('ไม่พบ Java -- กำลังติดตั้ง (ใช้เวลาราว 30 วินาที บน Colab)...')
    subprocess.run(['apt-get', '-qq', 'install', '-y',
                    'openjdk-17-jre-headless'], check=False)

print('java:', shutil.which('java') or 'ไม่พบ -- reasoning cell ด้านล่างอาจรันไม่ได้')

java: /usr/bin/java


In [3]:
from owlready2 import *

## To Create a SWRL Rule (ไม่มี builtin)

กฎ: *"หลานคือลูกของลูก"* — `hasParent(?x,?y) ^ hasParent(?y,?z) -> hasGrandparent(?x,?z)`

กฎที่ไม่มี `swrlb:` builtin รันได้ด้วย **HermiT** (เร็วกว่า ไม่ต้องใช้ Pellet)

In [4]:
onto = get_ontology("http://example.org/family#")
with onto:
    class Person(Thing): pass
    class hasParent(ObjectProperty): domain = [Person]; range = [Person]
    class hasGrandparent(ObjectProperty): domain = [Person]; range = [Person]

    rule = Imp()
    rule.set_as_rule(
        "hasParent(?x, ?y), hasParent(?y, ?z) -> hasGrandparent(?x, ?z)")

    john, mary, george = Person("john"), Person("mary"), Person("george")
    john.hasParent = [mary]
    mary.hasParent = [george]

sync_reasoner_hermit(onto.world, infer_property_values=True)
print("john hasGrandparent:", john.hasGrandparent)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmpr9jkbwht -Y


* Owlready * Adding relation family.john hasGrandparent family.george
john hasGrandparent: [family.george]


* Owlready2 * HermiT took 0.47106504440307617 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## SWRL Built-in ต้องใช้ Pellet

**HermiT ไม่รองรับ `swrlb:` built-in** — ต้องเรียก Pellet โดยตรงผ่าน `--loader OWLAPI` เพื่อเลี่ยงปัญหา Java version ของ jar ที่ owlready2 แถมมา

In [5]:
import glob, os, re, subprocess, tempfile

PELLET_DIR = os.path.join(os.path.dirname(owlready2.__file__), "pellet")

def run_pellet(onto, infer_values=True, timeout=120):
    with tempfile.NamedTemporaryFile("wb", suffix=".owl", delete=False) as tmp:
        path = tmp.name
    onto.save(path, format="rdfxml")
    jars = sorted(glob.glob(os.path.join(PELLET_DIR, "*.jar")))
    classpath = ":".join(jars + [PELLET_DIR])
    cmd = [owlready2.JAVA_EXE, "-Xmx1500M", "-cp", classpath,
           "pellet.Pellet", "realize", "--loader", "OWLAPI", "--ignore-imports"]
    if infer_values:
        cmd += ["--infer-prop-values", "--infer-data-prop-values"]
    cmd.append(path)
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    os.unlink(path)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[:800])
    return proc.stdout

print("run_pellet() พร้อมใช้งาน")

run_pellet() พร้อมใช้งาน


## รันกฎที่มี Built-in: คำนวณ BMI

`swrlb:multiply` และ `swrlb:divide` — **argument แรกคือผลลัพธ์เสมอ**

In [6]:
bmi_onto = World().get_ontology("http://example.org/health#")
with bmi_onto:
    class Patient(Thing): pass
    class Obese(Patient): pass
    class hasWeight(DataProperty, FunctionalProperty): domain=[Patient]; range=[float]
    class hasHeight(DataProperty, FunctionalProperty): domain=[Patient]; range=[float]
    class hasBMI(DataProperty, FunctionalProperty): domain=[Patient]; range=[float]

    r1 = Imp()
    r1.set_as_rule("Patient(?p), hasWeight(?p, ?w), hasHeight(?p, ?h), "
                   "multiply(?hh, ?h, ?h), divide(?bmi, ?w, ?hh) -> hasBMI(?p, ?bmi)")
    r2 = Imp()
    r2.set_as_rule("Patient(?p), hasBMI(?p, ?b), greaterThan(?b, 30.0) -> Obese(?p)")

    john = Patient("john")
    john.hasWeight, john.hasHeight = 95.0, 1.75

out = run_pellet(bmi_onto)
m = re.search(r"DATAPROPVAL:\s+\S*john\S*\s+\S*hasBMI\S*\s+literal\(([^,]+),", out)
print("john BMI =", m.group(1) if m else "(ไม่พบในผลลัพธ์)")
print("john เป็น Obese ไหม:", "Obese" in out)

john BMI = 31.02040816326530612244897959183673
john เป็น Obese ไหม: True


## 🎯 สรุป

- SWRL rule คือ `body -> head` (Horn clause) — ตัวแปรใน head ต้อง bind มาจาก body
- กฎที่**ไม่มี** `swrlb:` builtin รันได้ด้วย **HermiT** (เร็ว)
- กฎที่**มี** builtin ต้องใช้ **Pellet** — HermiT จะ error ทันที
- `swrlb:multiply(?result, ?a, ?b)` = `result = a * b` — **argument แรกคือผลลัพธ์เสมอ**
- เมื่อ `sync_reasoner_pellet()` พังเพราะ Java version ให้เรียก Pellet CLI ตรงด้วย `--loader OWLAPI`

ลองเพิ่มผู้ป่วยคนใหม่ที่มี weight/height ต่างกัน แล้วดูว่าใครถูกจัดเป็น Obese บ้าง!

---

## จบบทที่ 11 🎉

ไปต่อบทถัดไป: **บทที่ 12 — Linked Open Data**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*